In [1]:
#Fine tuning a pretrained model
# Scenario: Fine-tune ResNet-50 pretrained on ImageNet
#  to classify 5 types of plant disease from 3,000 leaf images.

# Import PyTorch main library
# PyTorch is used for building and training deep learning models
import torch

# Import torchvision which contains datasets, pretrained models and image
#  utilities
import torchvision

# Import neural network module from PyTorch
# nn contains layers like Linear, Conv2D, ReLU etc.
import torch.nn as nn

# Import pretrained models and image transformations
from torchvision import models, transforms

# Import DataLoader to load dataset in batches
from torch.utils.data import DataLoader

# Import ImageFolder dataset loader
# This loads images arranged in folder structure
# Example:
# dataset/
#     disease1/
#     disease2/
#     disease3/
from torchvision.datasets import ImageFolder


# ---------------------------------------------------------
# STEP 1 : Load a Pretrained ResNet-50 Model
# ---------------------------------------------------------


# ResNet50 is a deep CNN with 50 layers designed by Microsoft
# It is trained on ImageNet dataset (~1.2 million images, 1000 classes)


# Using pretrained weights helps the model already know
# basic visual features like edges, shapes, textures etc.

model = models.resnet50(weights='IMAGENET1K_V2')

# weights='IMAGENET1K_V2'
# loads pretrained weights trained on ImageNet
# This technique is called TRANSFER LEARNING

# Instead of training from scratch,
# we reuse knowledge learned from a huge dataset.


# ---------------------------------------------------------
# STEP 2 : Freeze ALL layers
# ---------------------------------------------------------

# Freezing means we stop updating weights during training

# Why freeze?
# Because the early layers already learned useful
# general visual features like:
# edges
# corners
# textures
# patterns

# We only want to train the final layers for our dataset.

for param in model.parameters():
    param.requires_grad = False

# requires_grad = False means:
# PyTorch will NOT compute gradients for these parameters
# during backpropagation.

# Result:
# The pretrained weights stay unchanged.


# ---------------------------------------------------------
# STEP 3 : Replace the classifier head (Fully Connected layer)
# ---------------------------------------------------------

# ResNet originally outputs 1000 classes (ImageNet classes)
# But our plant disease dataset has only 5 classes.

num_classes = 5

# The final layer of ResNet is:
# model.fc

# We replace it with a custom classifier.

model.fc = nn.Sequential(

    # Dropout randomly disables neurons during training
    # Helps reduce overfitting
    nn.Dropout(0.4),

    # First fully connected layer
    # model.fc.in_features gives the number of input features
    # from the previous ResNet layer (usually 2048)

    nn.Linear(model.fc.in_features, 256),

    # ReLU activation introduces non-linearity
    # Helps model learn complex patterns
    nn.ReLU(),

    # Final layer outputs probabilities for each class
    nn.Linear(256, num_classes)
)

# Architecture of new classifier:
#
# Feature Vector (2048)
#        ↓
#     Dropout
#        ↓
#   Linear (2048 → 256)
#        ↓
#       ReLU
#        ↓
#   Linear (256 → 5 classes)


# ---------------------------------------------------------
# STEP 4 : Unfreeze last residual block (Fine-Tuning)
# ---------------------------------------------------------

# ResNet architecture is divided into blocks:
#
# conv1
# layer1
# layer2
# layer3
# layer4
# fc

# layer4 is the deepest feature extraction block.

# We unfreeze it so the network can slightly adjust
# high-level features for plant diseases.

for param in model.layer4.parameters():
    param.requires_grad = True

# This technique is called:
# FINE-TUNING

# Strategy used here:
#
# Early layers  → Frozen (generic features)
# Deep layers   → Trainable (task-specific features)
# Classifier    → Fully trainable


# ---------------------------------------------------------
# STEP 5 : Differential Learning Rates
# ---------------------------------------------------------

# Different parts of the network learn at different speeds.

# Pretrained layers need SMALL learning rate
# because we only want small adjustments.

# New classifier layers need HIGHER learning rate
# because they are randomly initialized.

optimizer = torch.optim.Adam([

    # Lower learning rate for pretrained layer4
    {'params': model.layer4.parameters(), 'lr': 1e-4},

    # Higher learning rate for new classifier head
    {'params': model.fc.parameters(), 'lr': 1e-3},
])

# Adam optimizer:
# Adaptive learning rate optimization algorithm
# Combines benefits of Momentum and RMSProp

# Learning rates:
#
# layer4 → 0.0001 (small updates)
# fc     → 0.001  (faster learning)


# ---------------------------------------------------------
# STEP 6 : Loss Function
# ---------------------------------------------------------

# CrossEntropyLoss is used for multi-class classification.

criterion = nn.CrossEntropyLoss()

# It combines:
#
# LogSoftmax + Negative Log Likelihood

# Expected input:
#
# Model output → raw logits
# Target → class index (0,1,2,3,4)

# Example:
#
# Output = [2.1, -1.2, 0.5, 3.4, -0.7]
# Target = 3


# ---------------------------------------------------------
# FINAL TRAINING FLOW
# ---------------------------------------------------------

# During training the process is:

# 1️⃣ Input image
#        ↓
# 2️⃣ ResNet pretrained layers (mostly frozen)
#        ↓
# 3️⃣ Layer4 (fine-tuned)
#        ↓
# 4️⃣ Custom classifier (fc)
#        ↓
# 5️⃣ Output logits for 5 plant diseases
#        ↓
# 6️⃣ CrossEntropyLoss calculates error
#        ↓
# 7️⃣ Backpropagation
#        ↓
# 8️⃣ Only layer4 + fc weights update

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 173MB/s] 


In [ ]:
# Scenario: Your fine-tuned model hits 78% validation accuracy but
#  training accuracy is 96%. Classic overfitting. Here are your levers.
# ---------------------------------------------------------
# Import required libraries
# ---------------------------------------------------------

from torchvision import transforms      # Provides image preprocessing and augmentation utilities
import torch.optim as optim             # Contains optimization algorithms used during training
import torch                            # Core PyTorch library
import torch.nn as nn                   # Neural network layers and loss functions
import numpy as np                      # Used here to sample from Beta distribution for MixUp


# ---------------------------------------------------------
# DATA AUGMENTATION PIPELINE
# ---------------------------------------------------------

# Compose() chains multiple image transformations together
# Each transformation is applied sequentially to the input image
train_tf = transforms.Compose([

    # Randomly crop a region of the image and resize it to 224x224
    # scale=(0.7,1.0) means crop size will be between 70%–100% of the original image
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),

    # Randomly flip the image horizontally with 50% probability
    # Helps model learn orientation-invariant features
    transforms.RandomHorizontalFlip(p=0.5),

    # Randomly flip image vertically with 20% probability
    # Adds more variation in orientation
    transforms.RandomVerticalFlip(p=0.2),

    # Randomly change image brightness, contrast and saturation
    # Helps model generalize across lighting conditions
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2
    ),

    # Randomly rotate the image between -15° and +15°
    # Improves robustness to rotated objects
    transforms.RandomRotation(degrees=15),

    # Convert some images to grayscale with probability 0.1
    # Forces model to rely on shapes and textures rather than color only
    transforms.RandomGrayscale(p=0.1),

    # Convert image from PIL format to PyTorch tensor
    # Also scales pixel values from 0–255 to 0–1
    transforms.ToTensor(),

    # Normalize tensor using ImageNet mean and standard deviation
    # This is required when using pretrained ImageNet models
    transforms.Normalize(
        [0.485,0.456,0.406],   # mean for RGB channels
        [0.229,0.224,0.225]    # standard deviation for RGB channels
    ),

    # Randomly erase a rectangular region of the image
    # This technique is also called "Cutout regularization"
    # Prevents the model from focusing only on one region
    transforms.RandomErasing(p=0.2),

])


# ---------------------------------------------------------
# MIXUP DATA AUGMENTATION
# ---------------------------------------------------------

# MixUp creates new training samples by blending two images together
# This improves generalization and reduces overfitting

def mixup_data(x, y, alpha=0.4):

    # Sample mixing coefficient (lambda) from Beta distribution
    # Beta distribution ensures lambda is between 0 and 1
    lam = np.random.beta(alpha, alpha)

    # Randomly shuffle indices of the batch
    # This determines which image will be mixed with which
    idx = torch.randperm(x.size(0))

    # Create mixed images
    # New image = lam * imageA + (1-lam) * imageB
    mixed_x = lam * x + (1 - lam) * x[idx]

    # Return mixed images and both labels
    # Loss will be computed using both labels weighted by lambda
    return mixed_x, y, y[idx], lam


# ---------------------------------------------------------
# LEARNING RATE SCHEDULER
# ---------------------------------------------------------

# CosineAnnealingLR gradually reduces the learning rate
# following a cosine curve during training

scheduler = optim.lr_scheduler.CosineAnnealingLR(

    optimizer,      # optimizer whose learning rate will be scheduled

    T_max = 50,     # number of epochs before the learning rate reaches minimum

    eta_min = 1e-6  # minimum learning rate value

)

# Why use cosine annealing?
# • Smooth learning rate decay
# • Prevents sudden learning rate drops
# • Often improves convergence in deep learning models


# ---------------------------------------------------------
# LOSS FUNCTION WITH LABEL SMOOTHING
# ---------------------------------------------------------

# CrossEntropyLoss is commonly used for multi-class classification

criterion = nn.CrossEntropyLoss(

    label_smoothing = 0.1   # distributes a small probability mass to other classes

)

# Normally labels look like:
# [0,0,0,1,0]

# With label smoothing (0.1):
# [0.025,0.025,0.025,0.9,0.025]

# Benefits:
# • prevents overconfident predictions
# • improves generalization
# • reduces overfitting


# ---------------------------------------------------------
# GRADIENT CLIPPING
# ---------------------------------------------------------

# Gradient clipping prevents exploding gradients during backpropagation

torch.nn.utils.clip_grad_norm_(

    model.parameters(),   # parameters whose gradients will be clipped

    max_norm = 1.0        # maximum allowed gradient norm

)

# If gradient norm becomes larger than 1.0
# it will be scaled down automatically

# This stabilizes training especially in deep neural networks

tensor(0.)

In [ ]:
# Hyperparameter Tuning
# Scenario: You need to tune LR, batch size, dropout,
# and weight decay for your plant classifier. Manual search takes weeks. Optuna does it overnight.

# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import optuna
# Optuna is an automatic hyperparameter optimization framework.
# It runs multiple experiments (called trials) and searches for
# the best hyperparameter combination.

import torch
# Core PyTorch library used for tensor computation and deep learning.

import torch.nn as nn
# Contains neural network layers and loss functions.

import torch.nn.functional as F
# Functional API for neural network operations (activations, loss, etc.)

import torch.optim as optim
# Contains optimization algorithms such as Adam and SGD.

from torch.utils.data import DataLoader, TensorDataset
# DataLoader loads data in batches.
# TensorDataset wraps tensors so they can behave like a dataset.


# ============================================================
# 1. DEFINE A SIMPLE MODEL BUILDER FUNCTION
# ============================================================

def build_model(dropout=0.5):

    # This function constructs a neural network architecture.
    # The dropout parameter is supplied by Optuna so that
    # each trial can test a different dropout value.

    # The network below is a simple feedforward neural network
    # designed for MNIST-like data (28x28 grayscale images).

    model = nn.Sequential(

        nn.Linear(28*28, 128),
        # Fully connected layer
        # Input size = 28×28 pixels (flattened image)
        # Output size = 128 neurons

        nn.ReLU(),
        # ReLU activation function introduces non-linearity.
        # Formula: f(x) = max(0, x)

        nn.Dropout(dropout),
        # Randomly disables neurons during training.
        # Helps prevent overfitting.
        # Dropout value is chosen by Optuna.

        nn.Linear(128, 10)
        # Final classification layer.
        # Output size = 10 classes (digits 0–9).
    )

    return model
    # Return the constructed neural network.


# ============================================================
# 2. DEFINE TRAINING AND EVALUATION FUNCTION
# ============================================================

def train_and_evaluate(model, optimizer, batch_size):

    # This function performs:
    # 1. Training the model
    # 2. Evaluating validation accuracy

    # --------------------------------------------------------
    # Create a dummy dataset
    # --------------------------------------------------------

    X = torch.randn(500, 28*28)
    # Generate 500 random samples.
    # Each sample represents a flattened 28×28 image.

    y = torch.randint(0, 10, (500,))
    # Random labels between 0 and 9.

    dataset = TensorDataset(X, y)
    # Combines feature tensor X and label tensor y into a dataset.

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    # DataLoader loads data in batches.
    # batch_size is selected by Optuna.
    # shuffle=True randomizes data order each epoch.

    criterion = nn.CrossEntropyLoss()
    # Loss function used for multi-class classification.


    # --------------------------------------------------------
    # Training Phase
    # --------------------------------------------------------

    model.train()
    # Set model to training mode (enables dropout).

    for xb, yb in loader:

        optimizer.zero_grad()
        # Reset gradients from previous iteration.

        out = model(xb)
        # Forward pass through the neural network.

        loss = criterion(out, yb)
        # Compute classification loss.

        loss.backward()
        # Backpropagation: compute gradients.

        optimizer.step()
        # Update model parameters using optimizer.


    # --------------------------------------------------------
    # Validation Phase
    # --------------------------------------------------------

    model.eval()
    # Switch model to evaluation mode.
    # Disables dropout and other training-only layers.

    correct = 0

    with torch.no_grad():
        # Disable gradient computation during evaluation.

        for xb, yb in loader:

            preds = model(xb).argmax(dim=1)
            # Predict class with highest probability.

            correct += (preds == yb).sum().item()
            # Count correct predictions.

    val_acc = correct / len(dataset)
    # Compute validation accuracy.

    return val_acc
    # Return accuracy to Optuna.


# ============================================================
# 3. DEFINE OPTUNA OBJECTIVE FUNCTION
# ============================================================

def objective(trial):

    # The objective function defines one Optuna trial.
    # Each trial samples new hyperparameters and evaluates them.


    # --------------------------------------------------------
    # Sample Hyperparameters
    # --------------------------------------------------------

    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    # Learning rate sampled between 0.00001 and 0.01.
    # log=True samples values on logarithmic scale.

    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    # Batch size selected from predefined options.

    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    # Dropout rate between 0.1 and 0.5.

    weight_decay = trial.suggest_float('wd', 1e-5, 1e-2, log=True)
    # Weight decay (L2 regularization) parameter.

    optimizer_name = trial.suggest_categorical('optimizer', ['Adam','SGD'])
    # Select optimizer algorithm.


    # --------------------------------------------------------
    # Build Model
    # --------------------------------------------------------

    model = build_model(dropout=dropout)
    # Create neural network with sampled dropout.


    # --------------------------------------------------------
    # Create Optimizer
    # --------------------------------------------------------

    optimizer = getattr(optim, optimizer_name)(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # getattr dynamically selects optimizer:
    # If optimizer_name = "Adam"
    # → optim.Adam
    #
    # If optimizer_name = "SGD"
    # → optim.SGD


    # --------------------------------------------------------
    # Train Model and Compute Validation Accuracy
    # --------------------------------------------------------

    val_acc = train_and_evaluate(model, optimizer, batch_size)

    return val_acc
    # Optuna will try to maximize this value.


# ============================================================
# 4. CREATE AND RUN OPTUNA STUDY
# ============================================================

study = optuna.create_study(

    direction='maximize',
    # Indicates we want to maximize validation accuracy.

    pruner=optuna.pruners.MedianPruner()
    # Stops trials early if they perform worse than median
    # of previous trials. This speeds up optimization.
)


study.optimize(objective, n_trials=10)
# Runs hyperparameter optimization for 10 trials.
# Each trial tests a different hyperparameter configuration.


# ============================================================
# PRINT BEST RESULTS
# ============================================================

print('Best params:', study.best_params)
# Displays the best hyperparameter combination found.

print('Best val acc:', study.best_value)
# Displays the highest validation accuracy achieved.

[I 2026-03-08 12:32:36,136] A new study created in memory with name: no-name-f3b77ec2-5164-4bae-85dc-5be81e559b9b
[I 2026-03-08 12:32:36,230] Trial 0 finished with value: 0.846 and parameters: {'lr': 0.006817237695334311, 'batch_size': 16, 'dropout': 0.1266042296041323, 'wd': 0.00010062908043740897, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.846.
[I 2026-03-08 12:32:36,271] Trial 1 finished with value: 0.086 and parameters: {'lr': 1.813540852247543e-05, 'batch_size': 64, 'dropout': 0.25865445403238907, 'wd': 0.004446875591149855, 'optimizer': 'SGD'}. Best is trial 0 with value: 0.846.
[I 2026-03-08 12:32:36,334] Trial 2 finished with value: 0.808 and parameters: {'lr': 0.0019926197110337767, 'batch_size': 32, 'dropout': 0.14020495115798925, 'wd': 2.5028019007961012e-05, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.846.
[I 2026-03-08 12:32:36,375] Trial 3 finished with value: 0.096 and parameters: {'lr': 0.00029514370851054157, 'batch_size': 64, 'dropout': 0.467585200034

Best params: {'lr': 0.0032642406354698042, 'batch_size': 16, 'dropout': 0.12104449791605726, 'wd': 0.009360629531341981, 'optimizer': 'Adam'}
Best val acc: 0.898
